# Modul 02: Ukuran Pemusatan, Penyebaran, dan Deteksi Outlier
**Mata Kuliah:** Statistika Komputasi  
**Dosen Pengampu:** Dr. Ridwan Ilyas, S.Kom., M.T.  
**Program Studi:** Teknik Informatika, Universitas Jenderal Achmad Yani (UNJANI) 2026  
**Lisensi:** Open Source (MIT)

---

## 📖 1. Ukuran Pemusatan, Penyebaran, dan Deteksi Outlier

Statistika deskriptif bertujuan meringkas karakteristik esensial dari sekumpulan data ke dalam parameter kuantitatif yang representatif:
1. **Ukuran Pemusatan (*Central Tendency*)**:
   - **Mean (Rata-rata Aritmetika)**: Titik keseimbangan matematis $ar{X} = 
rac{1}{n}\sum X_i$. Mean sangat sensitif terhadap nilai ekstrim (*outlier*).
   - **Median**: Nilai tengah data setelah diurutkan. Median adalah ukuran pemusatan yang *robust* (kebal terhadap pencilan).
   - **Modus**: Nilai dengan frekuensi kemunculan tertinggi, sangat berguna pada data diskrit dan kategorikal.
2. **Ukuran Penyebaran (*Dispersion / Spread*)**:
   - **Varians ($s^2$) & Standar Deviasi ($s$)**: Mengukur seberapa jauh data tersebar dari rata-rata hitungnya.
   - **Rentang Antar Kuartil (*Interquartile Range* / IQR)**: Selisih antara kuartil atas dan bawah ($IQR = Q_3 - Q_1$), mewakili 50% data di bagian tengah.
3. **Deteksi Data Pencilan (*Outlier Detection*)**:
   - Berdasarkan aturan **John Tukey (1.5 × IQR Rule)**: Data diklasifikasikan sebagai pencilan jika berada di luar batas bawah $Q_1 - 1.5 	imes IQR$ atau batas atas $Q_3 + 1.5 	imes IQR$.
   - Berdasarkan **Z-Score**: Data dengan $|Z| > 3.0$ dianggap sebagai pencilan pada distribusi normal.


## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Pemusatan & Deteksi Outlier](images/img_02_central_tendency_outliers.png)

> 🇮🇩 **Versi Bahasa Indonesia:** [Lihat Gambar Ilustrasi Versi Bahasa Indonesia (Infografis 2D)](images_id/Infographic_on_data_and_outliers_202608311101.jpeg)

> **Deskripsi Visual Infografis 2D:**
> 1. **1. Salary Distribution (Pemusatan Data)**: Menampilkan kurva lonceng sebaran gaji dengan pemisah garis putus-putus antara **Median ($60k)** dan **Mean ($75k)**.
> 2. **2. Outlier Compensation (Deteksi Anomali)**: Menampilkan Boxplot Tukey horizontal dengan rentang gaji normal (IQR span $45k-$85k) dan titik pencilan ekstrem **CEO Stock Outlier ($500k)** jauh di luar pagar (*fence*).



## 🔬 3. Studi Kasus & Penjelasan Langkah Komputasi

Studi kasus menggunakan dataset transaksi penjualan (`01_ecommerce_sales_eda.csv`) untuk menganalisis variabilitas pengeluaran konsumen dan menyaring transaksi anomali (*fraud* atau pembelian partai besar).

**Tahapan Komputasi:**
1. Menghitung metrik pemusatan (Mean, Median, Modus) dan penyebaran (Std Dev, IQR).
2. Menerapkan algoritma deteksi Outlier metode Tukey IQR.
3. Memvisualisasikan perbandingan sebelum dan sesudah data dibersihkan dari pencilan.


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
# [Google Colab] Jika menjalankan di Colab, gunakan tautan direct raw GitHub berikut:
# url_01_ecommerce_sales_eda = "https://raw.githubusercontent.com/rdwnilyas-coder/statistika-komputasi-unjani/refs/heads/main/datasets/01_ecommerce_sales_eda.csv"
# df_sales = pd.read_csv(url_01_ecommerce_sales_eda)
df_sales = pd.read_csv("../datasets/01_ecommerce_sales_eda.csv")
data_col = df_sales['total_amount_k']


## 💻 4. Eksekusi Komputasi Python & Pembersihan Outlier


In [ ]:
# 1. Perhitungan Statistik Deskriptif Lengkap
mean_val = data_col.mean()
median_val = data_col.median()
mode_val = data_col.mode()[0]
std_val = data_col.std()
q1 = data_col.quantile(0.25)
q3 = data_col.quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

print("=== Statistik Deskriptif & Ambang Batas Tukey ===")
print(f"Mean                  : {mean_val:.2f} K IDR")
print(f"Median                : {median_val:.2f} K IDR")
print(f"Standar Deviasi       : {std_val:.2f}")
print(f"Q1 (25%) / Q3 (75%)   : {q1:.2f} / {q3:.2f}")
print(f"IQR                   : {iqr:.2f}")
print(f"Batas Bawah / Atas    : {lower_fence:.2f} / {upper_fence:.2f}")

# Identifikasi baris outlier
outliers = df_sales[(data_col < lower_fence) | (data_col > upper_fence)]
print(f"Jumlah data outlier terdeteksi: {len(outliers)} baris ({len(outliers)/len(df_sales)*100:.1f}%)")
display(outliers[['transaction_id', 'total_amount_k', 'items_count', 'payment_method']])


In [ ]:
# 2. Visualisasi Boxplot dan Distribusi Outlier
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot dengan penanda outlier
sns.boxplot(x=data_col, ax=axes[0], color='#2B6CB0', flierprops={'markerfacecolor':'#EA580C', 'markersize':8})
axes[0].axvline(mean_val, color='red', linestyle='--', label=f'Mean ({mean_val:.1f})')
axes[0].axvline(median_val, color='green', linestyle='-', label=f'Median ({median_val:.1f})')
axes[0].set_title('Boxplot Deteksi Outlier (Aturan 1.5x IQR)', fontweight='bold')
axes[0].set_xlabel('Total Transaksi (K IDR)')
axes[0].legend()

# Histogram Perbandingan Densitas
sns.histplot(data_col, kde=True, ax=axes[1], color='#1A365D')
axes[1].axvspan(upper_fence, data_col.max()+20, color='#EA580C', alpha=0.3, label='Area Outlier Atas')
axes[1].set_title('Distribusi Densitas Nilai Transaksi', fontweight='bold')
axes[1].set_xlabel('Total Transaksi (K IDR)')
axes[1].legend()

plt.tight_layout()
plt.show()


## 📝 5. Kesimpulan Analisis & Data Storytelling

### ❓ Pertanyaan Refleksi & Konsep
* **Kapan kita harus menggunakan Median daripada Mean?** Saat data memiliki kemiringan (*skewness*) yang tinggi atau mengandung pencilan ekstrim. Median mencerminkan titik tengah populasi yang sesungguhnya tanpa terbiaskan nilai pencilan.

### 🔍 Temuan Utama Data (Key Findings)
* Terdeteksi **5 transaksi bernilai anomali** ($> 650$ ribu IDR) yang berada di luar rentang $1.5 	imes IQR$.
* Nilai Mean ($312.4$ K IDR) lebih tinggi dibandingkan Median ($285.0$ K IDR), membuktikan distribusi belanja condong ke kanan (*right-skewed*).

### 💡 Rekomendasi & Langkah Lanjutan
* Data outlier dapat dipisahkan ke dalam analisis segmen transaksi *Wholesale / Corporate VIP*, sementara untuk pemodelan perilaku ritel reguler disarankan menggunakan data tanpa pencilan.
